# Separable CMA-ES

A ready-to-run CPU example of the same creator/evaluator/callback contract used
by GA and NSGA. Install the checkout with `pip install -e ".[cma]"` or
`uv sync --extra cma` first. This example needs no model weights or GPU.

1. Configure a normalized search space and deterministic seed.
2. Compose a creator, maximization evaluator and generation callback.
3. Run and inspect the best candidate and generation fitness.

For SDXL embeddings, use `EmbeddingCoordinates.encode` for the initial mean and
pass `EmbeddingCoordinates.decode` to the algorithm. See
[configuration and checkpoint behavior](../docs/sep_cma.md).

In [ ]:
from pathlib import Path
import sys

checkout = next(path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / "evolutionary/algorithms/sep_cma.py").is_file())
sys.path.insert(0, str(checkout))

import numpy as np
from evolutionary.algorithms.sep_cma import SepCMAES
from evolutionary.evolution_base import SolutionCandidate, SolutionCreator, SingleObjectiveEvaluator

The optimizer searches coordinates in `[0, 1]`. The creator here simply
copies those coordinates into the result. The evaluator returns negative squared
distance, so the framework's usual maximization convention finds the target.

In [ ]:
population_size = 16
num_generations = 30
seed = 42
sigma = 0.15
initial_mean = np.full(4, 0.5)
target = np.array([0.2, 0.4, 0.7, 0.9])

class VectorCreator(SolutionCreator):
    def create_solution(self, argument):
        return SolutionCandidate(argument, argument.copy())

class TargetEvaluator(SingleObjectiveEvaluator):
    def evaluate(self, result):
        return -float(np.sum((result - target) ** 2))

history = []
def record_generation(generation, algorithm):
    history.append((generation, algorithm.best_solution().fitness))

Every generation evaluates one fresh population. The initial mean centers
the first proposals; it is not itself a scored candidate. Native pycma minimizes,
so the adapter reverses fitness before `tell` and retains the best candidate from
all evaluated populations.

In [ ]:
algorithm = SepCMAES(
    num_generations=num_generations,
    population_size=population_size,
    solution_creator=VectorCreator(),
    evaluator=TargetEvaluator(),
    initial_mean=initial_mean,
    seed=seed,
    sigma=sigma,
    post_evaluation_callback=record_generation,
)
best = algorithm.run()
print("Best coordinates:", np.round(best.arguments, 4))
print("Squared distance:", -best.fitness)
print("Fresh evaluations:", algorithm.evaluation_count)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(*zip(*history))
plt.xlabel("Generation")
plt.ylabel("Best fitness so far")
plt.show()

Try a different target or seed, then rerun from the configuration cell.
For a larger domain, provide an ordered batch creator/evaluator, or use the
`SepCMAOptimizer` ask/tell interface when the application owns persistence.
A non-finite or incomplete score population is rejected without advancing its
budget. Keep experiment-specific calibration, score journals and held-out work
in the experiment package.